In [ ]:
import csv
import os
import numpy as np
import torch
from torch.utils.data import Dataset

class Gesture3DDataset(Dataset):
    def __init__(self, csv_file, split="train"):
        """
        csv_file: path to your CSV with [filepath,label,split]
        split: one of ["train", "val", "test"]
        """
        self.samples = []
        
        # 1. Read CSV, store (filepath, label) only for this `split`
        with open(csv_file, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["split"] == split:
                    self.samples.append((row["filepath"], row["label"]))

        # 2. Build a label_to_idx dict so we can convert label strings -> int
        all_labels = sorted(list(set(s[1] for s in self.samples)))
        self.label_to_idx = {lab: i for i, lab in enumerate(all_labels)}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filepath, label_str = self.samples[idx]
        
        # 3. Load .npy array of shape (120, 64, 64)
        arr = np.load(filepath)  # shape: (T=120, H=64, W=64), dtype=uint8

        # 4. Add channel dimension => (1, 120, 64, 64)
        # Because PyTorch 3D conv typically expects (C, D, H, W).
        # Here D=120 is 'time', H=64, W=64, and we have 1 channel (grayscale).
        arr = np.expand_dims(arr, axis=0)  # shape (1, 120, 64, 64)

        # Convert to float, optional normalization [0..1] if you like:
        arr = arr.astype(np.float32) / 255.0  # scale to [0..1]

        # 5. Convert to torch tensor
        x = torch.from_numpy(arr)  # shape: (1, 120, 64, 64)

        # 6. Convert label string -> int
        label_idx = self.label_to_idx[label_str]
        y = torch.tensor(label_idx, dtype=torch.long)

        return x, y


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Simple3DCNN(nn.Module):
    """
    A very basic 3D CNN that expects input of shape:
      (batch_size, 1, 120, 64, 64)
    You might need to adjust pool sizes if you run out of dims.
    """
    def __init__(self, num_classes=20):
        super(Simple3DCNN, self).__init__()
        
        # 1st conv block
        self.conv1 = nn.Conv3d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm3d(32)
        self.pool1 = nn.MaxPool3d(kernel_size=(2, 2, 2))  # reduce time, height, width by factor of 2

        # 2nd conv block
        self.conv2 = nn.Conv3d(32, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm3d(64)
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2))

        # 3rd conv block
        self.conv3 = nn.Conv3d(64, 128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm3d(128)
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2))

        # After 3 layers of pooling, we drastically reduce T, H, W.
        # Let's see: 120 -> 60 -> 30 -> 15 in time dimension,
        # and 64 -> 32 -> 16 -> 8 in height/width.

        # We'll flatten and use a fully-connected layer
        # 128 * 15 * 8 * 8 = 128 * 15 * 64 * 8 = 128 * 7680 ... that might still be big
        # Let's do an adaptive avg pool so we don't have to manually compute
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))  # this squashes all dims to 1

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        # shape (batch, 1, 120, 64, 64)
        x = self.pool1(self.bn1(F.relu(self.conv1(x))))  # => (batch, 32, 60, 32, 32)
        x = self.pool2(self.bn2(F.relu(self.conv2(x))))  # => (batch, 64, 30, 16, 16)
        x = self.pool3(self.bn3(F.relu(self.conv3(x))))  # => (batch, 128, 15, 8, 8)

        # adaptive avg pool => (batch, 128, 1, 1, 1)
        x = self.avgpool(x)
        # flatten
        x = x.view(x.size(0), -1)  # shape (batch, 128)
        # classify
        x = self.fc(x)            # shape (batch, num_classes)
        return x


In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

def train_3d_cnn(csv_file, num_classes=20, epochs=10, batch_size=2, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Create datasets
    train_dataset = Gesture3DDataset(csv_file, split="train")
    val_dataset   = Gesture3DDataset(csv_file, split="val")
    test_dataset  = Gesture3DDataset(csv_file, split="test")

    # 2. DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # 3. Instantiate model, loss, optimizer
    model = Simple3DCNN(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 4. Training loop
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)  # shape (batch, num_classes)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
        
        train_acc = correct / total
        avg_loss = running_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss_sum = 0.0
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for val_inputs, val_targets in val_loader:
                val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)
                val_outputs = model(val_inputs)
                val_loss = criterion(val_outputs, val_targets)
                val_loss_sum += val_loss.item()

                _, val_preds = torch.max(val_outputs, 1)
                val_correct += (val_preds == val_targets).sum().item()
                val_total += val_targets.size(0)
        
        val_acc = val_correct / val_total
        val_avg_loss = val_loss_sum / len(val_loader)

        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {avg_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_avg_loss:.4f}, Val Acc: {val_acc*100:.2f}%")

    # 5. Final test evaluation
    model.eval()
    test_loss_sum = 0.0
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for test_inputs, test_targets in test_loader:
            test_inputs, test_targets = test_inputs.to(device), test_targets.to(device)
            test_outputs = model(test_inputs)
            t_loss = criterion(test_outputs, test_targets)
            test_loss_sum += t_loss.item()
            _, test_preds = torch.max(test_outputs, 1)
            test_correct += (test_preds == test_targets).sum().item()
            test_total += test_targets.size(0)
    test_acc = test_correct / test_total
    test_avg_loss = test_loss_sum / len(test_loader)

    print(f"Test Loss: {test_avg_loss:.4f}, Test Accuracy: {test_acc*100:.2f}%")

    # Save model
    torch.save(model.state_dict(), "simple_3dcnn_gray64_20_epoch.pth")
    print("Model saved to simple_3dcnn_gray64_20_epoch.pth")

    return model


In [ ]:
if __name__ == "__main__":
    CSV_FILE = "dataset_split.csv" 
    trained_model = train_3d_cnn(
        csv_file=CSV_FILE,
        num_classes=20,  
        epochs=20,
        batch_size=2,
        lr=1e-3
    )


Epoch [1/20] | Train Loss: 2.9765, Train Acc: 7.14% | Val Loss: 2.8043, Val Acc: 6.67%
Epoch [2/20] | Train Loss: 2.8322, Train Acc: 12.14% | Val Loss: 2.6578, Val Acc: 11.67%
Epoch [3/20] | Train Loss: 2.7462, Train Acc: 16.79% | Val Loss: 2.4404, Val Acc: 36.67%
Epoch [4/20] | Train Loss: 2.5380, Train Acc: 22.14% | Val Loss: 2.6350, Val Acc: 16.67%
Epoch [5/20] | Train Loss: 2.4363, Train Acc: 25.36% | Val Loss: 2.1755, Val Acc: 31.67%
Epoch [6/20] | Train Loss: 2.3812, Train Acc: 29.64% | Val Loss: 2.1803, Val Acc: 26.67%
Epoch [7/20] | Train Loss: 2.2571, Train Acc: 31.07% | Val Loss: 1.9384, Val Acc: 36.67%
Epoch [8/20] | Train Loss: 2.1223, Train Acc: 37.86% | Val Loss: 2.3297, Val Acc: 23.33%
Epoch [9/20] | Train Loss: 1.9753, Train Acc: 45.71% | Val Loss: 1.5625, Val Acc: 41.67%
Epoch [10/20] | Train Loss: 1.7795, Train Acc: 48.21% | Val Loss: 1.5346, Val Acc: 51.67%
Epoch [11/20] | Train Loss: 1.7579, Train Acc: 53.21% | Val Loss: 1.6794, Val Acc: 46.67%
Epoch [12/20] | Train Loss: 1.5457, Train Acc: 65.36% | Val Loss: 1.3821, Val Acc: 53.33%
Epoch [13/20] | Train Loss: 1.4180, Train Acc: 68.57% | Val Loss: 1.5849, Val Acc: 50.00%
Epoch [14/20] | Train Loss: 1.2804, Train Acc: 67.86% | Val Loss: 1.5320, Val Acc: 48.33%
Epoch [15/20] | Train Loss: 1.1446, Train Acc: 76.43% | Val Loss: 0.9594, Val Acc: 66.67%
Epoch [16/20] | Train Loss: 0.9891, Train Acc: 80.71% | Val Loss: 2.2271, Val Acc: 33.33%
Epoch [17/20] | Train Loss: 0.9857, Train Acc: 81.07% | Val Loss: 1.2464, Val Acc: 53.33%
Epoch [18/20] | Train Loss: 0.7566, Train Acc: 87.86% | Val Loss: 0.8011, Val Acc: 70.00%
Epoch [19/20] | Train Loss: 0.6634, Train Acc: 89.64% | Val Loss: 0.7113, Val Acc: 85.00%
Epoch [20/20] | Train Loss: 0.6163, Train Acc: 90.00% | Val Loss: 0.4085, Val Acc: 96.67%
Test Loss: 0.3712, Test Accuracy: 96.67%
Model saved to simple_3dcnn_gray64_20_epoch.pth